## OSM Street Number Downloader
Downloads addr:housenumber data from OpenStreetMap for major European cities
using the Overpass API. Saves results as GeoJSON and CSV.

In [3]:
import requests
import json
import csv
import time
import os
from pathlib import Path
from tqdm import tqdm

OVERPASS_URLS = [
    "https://overpass-api.de/api/interpreter",
    "https://lz4.overpass-api.de/api/interpreter",
    "https://z.overpass-api.de/api/interpreter",
]
OVERPASS_MIRRORS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]

OUTPUT_DIR = Path("data/osm_adresses")

In [4]:
import os
os.chdir("/Users/terezasaskova/Library/Mobile Documents/com~apple~CloudDocs/3-30-300-index")


In [ ]:
# Major European cities with their OSM relation IDs
# relation ID = the OSM administrative boundary (faster & more accurate than name search)
CITIES = {
    #"Berlin":       {"relation": 62422,   "country": "DE"},
    #"London":       {"relation": 65606,   "country": "GB"},
    #"Paris":        {"relation": 71525,   "country": "FR"},
    #"Madrid":       {"relation": 5326784, "country": "ES"},
    #"Rome":         {"relation": 41485,   "country": "IT"},
    #"Vienna":       {"relation": 109166,  "country": "AT"},
    #"Warsaw":       {"relation": 336075,  "country": "PL"},
    #"Budapest":     {"relation": 3600170, "country": "HU"},
    #"Amsterdam":    {"relation": 271110,  "country": "NL"},
    #"Prague":       {"relation": 435514,  "country": "CZ"},
    #"Brussels":     {"relation": 58004,   "country": "BE"},
    #"Stockholm":    {"relation": 398021,  "country": "SE"},
    #"Munich":       {"relation": 62428,   "country": "DE"},
    #"Hamburg":      {"relation": 62782,   "country": "DE"},
    "Bucharest":    {"relation": 1246768, "country": "RO"},
    #"Sofia":        {"relation": 1683845, "country": "BG"},
    #"Copenhagen":   {"relation": 2192363, "country": "DK"},
    #"Helsinki":     {"relation": 34914,   "country": "FI"},
    #"Oslo":         {"relation": 406091,  "country": "NO"},
    #"Zurich":       {"relation": 1682248, "country": "CH"},
    #"Lisbon":       {"relation": 5690595, "country": "PT"},
    #"Athens":       {"relation": 6306116, "country": "GR"},
    #"Zagreb":       {"relation": 3083947, "country": "HR"},
    #"Bratislava":   {"relation": 14296293,"country": "SK"},
    #"Ljubljana":    {"relation": 2091468, "country": "SI"},
}


In [6]:
def build_query(relation_id):
    return f"""
[out:json][timeout:300];
area(id:{relation_id + 3600000000})->.a;
node["addr:housenumber"](area.a);
out body;
"""

def fetch_overpass(query: str, retries: int = 3, backoff: float = 10.0) -> dict | None:
    for mirror in OVERPASS_MIRRORS:
        for attempt in range(1, retries + 1):
            try:
                resp = requests.post(
                    mirror,
                    data={"data": query},
                    timeout=240,
                    headers={"User-Agent": "OSM-Address-Downloader/1.0"},
                )
                if resp.status_code == 200:
                    return resp.json()
                elif resp.status_code == 429:
                    wait = backoff * attempt
                    print(f"  Rate-limited by {mirror}. Waiting {wait}s …")
                    time.sleep(wait)
                else:
                    print(f"  HTTP {resp.status_code} from {mirror} (attempt {attempt})")
                    time.sleep(backoff)
            except requests.exceptions.Timeout:
                print(f"  Timeout on {mirror} (attempt {attempt})")
                time.sleep(backoff)
            except requests.exceptions.RequestException as e:
                print(f"  Request error: {e}")
                time.sleep(backoff)
        print(f"  All retries failed for mirror {mirror}, trying next …")
    return None

def parse_elements(elements: list) -> list[dict]:
    records = []
    for el in elements:
        tags = el.get("tags", {})
        housenumber = tags.get("addr:housenumber", "").strip()
        if not housenumber:
            continue

        # Geometry: nodes have lat/lon directly; ways expose a 'center'
        if el["type"] == "node":
            lat, lon = el.get("lat"), el.get("lon")
        elif el["type"] == "way":
            center = el.get("center", {})
            lat, lon = center.get("lat"), center.get("lon")
        else:
            continue

        records.append({
            "osm_type":   el["type"],
            "osm_id":     el["id"],
            "lat":        lat,
            "lon":        lon,
            "housenumber": housenumber,
            "street":     tags.get("addr:street", ""),
            "postcode":   tags.get("addr:postcode", ""),
            "city":       tags.get("addr:city", ""),
            "suburb":     tags.get("addr:suburb", ""),
            "country":    tags.get("addr:country", ""),
        })
    return records


In [7]:

def save_csv(records: list[dict], path: Path):
    if not records:
        return
    fieldnames = list(records[0].keys())
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(records)


def save_geojson(records: list[dict], path: Path):
    features = []
    for r in records:
        if r["lat"] is None or r["lon"] is None:
            continue
        features.append({
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [r["lon"], r["lat"]],
            },
            "properties": {k: v for k, v in r.items() if k not in ("lat", "lon")},
        })
    geojson = {"type": "FeatureCollection", "features": features}
    with open(path, "w", encoding="utf-8") as f:
        json.dump(geojson, f, ensure_ascii=False, separators=(",", ":"))


# ── Main ──────────────────────────────────────────────────────────────────────

def download_city(name: str, info: dict) -> int:
    relation_id = info["relation"]
    city_dir = OUTPUT_DIR / name
    city_dir.mkdir(parents=True, exist_ok=True)

    csv_path     = city_dir / f"{name}_addresses.csv"
    geojson_path = city_dir / f"{name}_addresses.geojson"

    # Skip if already downloaded
    if csv_path.exists():
        print(f"  ✓ {name} already downloaded — skipping")
        return -1

    print(f"  Querying Overpass for {name} (relation {relation_id}) …")
    query  = build_query(relation_id)
    result = fetch_overpass(query)

    if result is None:
        print(f"  ✗ Failed to fetch {name}")
        return 0

    elements = result.get("elements", [])
    records  = parse_elements(elements)

    save_csv(records, csv_path)
    save_geojson(records, geojson_path)

    return len(records)



In [8]:

def main(cities: dict = None):
    cities = cities or CITIES
    OUTPUT_DIR.mkdir(exist_ok=True)

    summary = []
    print(f"\n{'─'*55}")
    print(f"  OSM Street Number Downloader — {len(cities)} cities")
    print(f"{'─'*55}\n")

    for name, info in tqdm(cities.items(), desc="Cities", unit="city"):
        print(f"\n[{name}]")
        count = download_city(name, info)
        summary.append((name, info["country"], count))
        if count > 0:
            print(f"  → {count:,} address records saved")
        # Be polite to the Overpass API
        time.sleep(5)

    # Write summary
    summary_path = OUTPUT_DIR / "summary.csv"
    with open(summary_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["city", "country", "address_count"])
        for row in summary:
            if row[2] >= 0:
                writer.writerow(row)

    print(f" Summary: {summary_path.resolve()}")



In [17]:
main()  


───────────────────────────────────────────────────────
  OSM Street Number Downloader — 1 cities
───────────────────────────────────────────────────────



Cities:   0%|          | 0/1 [00:00<?, ?city/s]


[Bucharest]
  Querying Overpass for Bucharest (relation 1246768) …
  HTTP 504 from https://overpass-api.de/api/interpreter (attempt 1)
  HTTP 504 from https://overpass-api.de/api/interpreter (attempt 2)


Cities: 100%|██████████| 1/1 [00:44<00:00, 44.94s/city]

 Summary: /Users/terezasaskova/Library/Mobile Documents/com~apple~CloudDocs/3-30-300-index/data/osm_adresses/summary.csv


In [ ]:
import requests
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

BASE_URL = "https://geoportale.comune.milano.it/arcgis/rest/services/Toponomastica/Civici_OpenData/MapServer/0/query"

params = {
    "where": "1=1",
    "outFields": "*",
    "f": "json",
    "outSR": "4326",
    "resultOffset": 0,
    "resultRecordCount": 2000,
}

all_features = []
while True:
    r = requests.get(BASE_URL, params=params)
    data = r.json()
    features = data.get("features", [])
    all_features.extend(features)
    print(f"Downloaded {len(all_features)} records so far...")
    
    # Stop if fewer than 2000 returned — means we hit the last page
    if len(features) < 2000:
        break
    params["resultOffset"] += 2000

# Convert to GeoDataFrame
records = []
for f in all_features:
    attrs = f["attributes"]
    x = f["geometry"]["x"]
    y = f["geometry"]["y"]
    attrs["geometry"] = Point(x, y)
    records.append(attrs)

gdf = gpd.GeoDataFrame(records, geometry="geometry", crs="EPSG:4326")
print(f"\nTotal: {len(gdf):,} civici")

gdf.to_file("milano_civici.gpkg", driver="GPKG")
print("Saved to milano_civici.gpkg")